# Workshop 2: Regresión con Redes Neuronales Convolucionales (PyTorch)

**Objetivo**: Entrenar una red neuronal profunda que pueda estimar la edad (un valor numérico continuo) a partir de una fotografía facial. 

Utilizaremos el dataset **UTKFace**, que contiene más de 20,000 imágenes con etiquetas de edad, género y etnia.

## 1. Configuración del Entorno e Importaciones

In [ ]:
import os, re, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# PyTorch: Framework principal de Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import r2_score

# --- CONFIGURACIÓN DE HIPERPARÁMETROS --- 
# Definimos constantes globales para facilitar experimentos posteriores
DATA_DIR = Path("dataset")
IMG_SIZE = 64      # Tamaño al que redimensionaremos las fotos (64x64px)
BATCH_SIZE = 64    # Número de imágenes que procesará la red a la vez
NUM_EPOCHS = 20    # Cuántas veces el modelo verá todo el dataset
LR = 1e-3          # Tasa de aprendizaje inicial (Learning Rate)
SEED = 42          # Semilla para que los resultados sean reproducibles

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- SELECCIÓN DE HARDWARE ---
# Intentamos usar GPU (NVIDIA CUDA) para acelerar el entrenamiento 10-50 veces más que una CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Hardware detectado: {DEVICE}")

## 2. Definición del Dataset y Transformaciones

En PyTorch, necesitamos una clase `Dataset` que le diga al modelo cómo leer los archivos del disco y convertirlos en Tensores (matrices numéricas).

In [ ]:
class AgeDataset(Dataset):
    """Clase personalizada para manejar el dataset UTKFace."""
    def __init__(self, root_dir: Path, transform=None):
        self.transform, self.samples = transform, []
        # Recorremos la carpeta buscando imágenes
        for img_path in root_dir.iterdir():
            if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                # UTKFace nombra archivos como: [edad]_[género]_[etnia]_[fecha].jpg
                # Usamos Regex para extraer el primer número (la edad)
                match = re.match(r'^(\d+)_', img_path.name)
                if match:
                    age = int(match.group(1))
                    # Solo agregamos muestras válidas
                    if 0 <= age <= 116: 
                        self.samples.append((img_path, float(age)))

    def __len__(self): 
        # Retorna el tamaño total del dataset
        return len(self.samples)

    def __getitem__(self, idx):
        # Método que carga una imagen individual por su índice
        path, age = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: 
            img = self.transform(img)
        # Retornamos (Imagen en tensor, Edad como float32)
        return img, torch.tensor(age, dtype=torch.float32)

# --- PIPELINE DE TRANSFORMACIONES --- 
# La normalización con IMAGENET_MEAN ayuda a que el modelo converja más rápido 
# al usar estadísticas de millones de fotos pre-entrenadas.
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Transformaciones de ENTRENAMIENTO: Incluyen aumento de datos (Data Augmentation)
train_tf = transforms.Compose([
    transforms.Resize((64,64)), 
    transforms.RandomHorizontalFlip(p=0.5), # Gira la foto espejo para que el modelo no dependa de la orientación
    transforms.RandomRotation(10),           # Rota un poco la cabeza para simular ángulos reales
    transforms.ToTensor(),                   # Convierte píxeles [0,255] a rango [0,1]
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD) # Estandariza la distribución de color
])

# Transformaciones de PRUEBA: Solo redimensionar y normalizar (nada aleatorio)
val_tf = transforms.Compose([
    transforms.Resize((64,64)), 
    transforms.ToTensor(), 
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

train_ds = AgeDataset(DATA_DIR/"train", train_tf); 
val_ds = AgeDataset(DATA_DIR/"val", val_tf)

# DataLoader: Genera lotes (batches) automáticos. 
# Shuffle=True en entrenamiento evita que la red aprenda el orden de las fotos.
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False)

print(f"[!] Dataset listo! Entrenamiento: {len(train_ds)} imgs | Validación: {len(val_ds)} imgs")

## 3. Análisis Exploratorio de Datos (EDA)

Aquí visualizamos la distribución de edades para entender si nuestro dataset tiene "huecos" (por ejemplo, pocas fotos de ancianos o bebés).

In [ ]:
ages = [age for _, age in train_ds.samples]
plt.hist(ages, bins=20, color='skyblue', edgecolor='black')
plt.title("Distribución de Edades en Entrenamiento")
plt.xlabel("Edad"); plt.ylabel("Frecuencia")
plt.show()

## 4. Arquitectura del Modelo (CNN)

Nuestra red (AgeCNN) tiene dos partes:
1. **Extractor de características**: Capas convolucionales que detectan bordes, ojos, narices, etc.
2. **Regresor**: Capas Densas que convierten esas formas en un número final (la edad).

In [ ]:
class AgeCNN(nn.Module):
    """Red Neuronal Convolucional (CNN) para Estimación de Edad."""
    def __init__(self):
        super().__init__()
        
        # Bloque reutilizable: Conv -> BatchNorm -> ReLU -> MaxPool
        def block(in_c, out_c): 
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1), # Detecta patrones locales
                nn.BatchNorm2d(out_c), # Mantiene los valores en rangos saludables para la red
                nn.ReLU(),             # Introduce no-linealidad (aprender curvas)
                nn.MaxPool2d(2)        # Reduce el tamaño espacial (64x64 -> 32x32, etc.)
            )
        
        # 1. Capas Convolucionales: Cada bloque reduce la imagen y aumenta la profundidad (canales)
        self.feats = nn.Sequential(
            block(3, 32),   # Salida: 32 canales x 32x32px
            block(32, 64),  # Salida: 64 canales x 16x16px
            block(64, 128), # Salida: 128 canales x 8x8px
            block(128, 256) # Salida: 256 canales x 4x4px
        )
        
        # 2. Capas de Clasificación (Regresión):
        self.regr = nn.Sequential(
            nn.Flatten(),             # Aplasta la matriz 256x4x4 en un vector de 4096 elementos
            nn.Linear(256*4*4, 512),  # Conecta todo a 512 neuronas
            nn.BatchNorm1d(512), 
            nn.ReLU(), 
            nn.Dropout(0.5),          # Apaga al 50% de las neuronas al azar para evitar sobreajuste
            nn.Linear(512, 1)          # Salida final: 1 único número (la Edad)
        )
        
    def forward(self, x): 
        # Flujo: Entrada -> Convoluciones -> Regresión -> Resultado
        x = self.feats(x)
        return self.regr(x)

model = AgeCNN().to(DEVICE)
print("[*] Modelo creado satisfactoriamente!")

## 5. Entrenamiento del Modelo

Definimos qué tan duro se castigará el error (MSE) y cómo la red ajustará sus "perillas" (Adam).

In [ ]:
# CRITERIO: Usamos Error Cuadrático Medio (MSE) porque queremos predecir números, no categorías.
criterion = nn.MSELoss() 

# OPTIMIZADOR: Adam ajusta automáticamente la intensidad de los cambios en los pesos.
opt = optim.Adam(model.parameters(), lr=1e-3) 

# SCHEDULER: Si la red deja de mejorar, reducimos el Learning Rate a la mitad (factor=0.5)
# para que el ajuste sea más 'fino' al final.
sch = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=3)

hist = {"tl":[], "vl":[], "tm":[], "vm":[]}
for e in range(1, NUM_EPOCHS + 1):
    # --- FASE DE ENTRENAMIENTO ---
    model.train(); tl, tm = 0, 0
    for imgs, ages in tqdm(train_loader, desc=f"Epoch {e}"): 
        imgs, ages = imgs.to(DEVICE), ages.to(DEVICE)
        
        opt.zero_grad()        # Reseteamos gradientes anteriores
        p = model(imgs).squeeze() # Hacemos la predicción
        loss = criterion(p, ages) # Calculamos error matemático (MSE)
        loss.backward()        # Propagamos el error hacia atrás (Backprop)
        opt.step()             # Ajustamos los pesos
        
        tl += loss.item(); 
        tm += torch.abs(p - ages).mean().item() # Error Absoluto Medio (en años)
    
    # --- FASE DE VALIDACIÓN ---
    model.eval(); vl, vm = 0, 0
    with torch.no_grad(): # Desactivamos gradientes para ahorrar memoria
        for imgs, ages in val_loader: 
            imgs, ages = imgs.to(DEVICE), ages.to(DEVICE)
            p = model(imgs).squeeze(); 
            vl += criterion(p, ages).item(); 
            vm += torch.abs(p - ages).mean().item()
    
    # Guardamos historia para las gráficas
    avg_vl = vl/len(val_loader)
    hist["tl"].append(tl/len(train_loader)); hist["vl"].append(avg_vl)
    hist["tm"].append(tm/len(train_loader)); hist["vm"].append(vm/len(val_loader))
    
    sch.step(avg_vl) # Le avisamos al scheduler si mejoramos o no
    print(f"[*] Época {e} -> T-MAE: {tm/len(train_loader):.2f} | V-MAE: {vm/len(val_loader):.2f}")

# Guardamos el cerebro de la red entrenada
torch.save(model.state_dict(), "best_age_model.pth")

### Interpretación de Resultados
- Si la curva azul (Train) es mucho más baja que la naranja (Val), tenemos **Overfitting**.
- El punto mínimo de la curva de Validación indica el mejor estado de nuestro modelo.

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.plot(hist['tl'], label='Loss Ent.'); plt.plot(hist['vl'], label='Loss Val.'); plt.title('Criterio MSE'); plt.legend()
plt.subplot(1,2,2); plt.plot(hist['tm'], label='MAE Ent.'); plt.plot(hist['vm'], label='MAE Val.'); plt.title('Error en Años (MAE)'); plt.legend()
plt.show()

## 6. Evaluación Final y Coeficiente R2

El R2 Score nos dice qué porcentaje de la variabilidad de los datos explica el modelo. 1.0 es perfecto, 0.0 es tan bueno como tirar una moneda.

In [ ]:
model.eval(); preds, truths = [], []
with torch.no_grad():
    for imgs, ages in val_loader:
        p = model(imgs.to(DEVICE)).squeeze()
        # Guardamos todas las predicciones para calcular métricas finales
        preds.extend(p.cpu().numpy()); truths.extend(ages.numpy())

final_mae = np.mean(np.abs(np.array(preds) - np.array(truths)))
r2 = r2_score(truths, preds)
print(f"-- RESULTADOS FINALES --")
print(f" Error promedio: {final_mae:.1f} años")
print(f" Coeficiente R2: {r2:.4f}")

## 8. Inferencia Manual (Ver el modelo en acción)

In [ ]:
# Elegimos una imagen al azar del conjunto de validación
idx = random.randint(0, len(val_ds)-1)
img_t, age = val_ds[idx]

model.eval()
with torch.no_grad():
    # Agregamos una dimensión de batch (1, 3, 64, 64) con unsqueeze(0)
    p_age = model(img_t.unsqueeze(0).to(DEVICE)).item()

# Deshacemos la normalización para que la imagen se vea bien (colores naturales)
inv_img = img_t.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
plt.imshow(np.clip(inv_img, 0, 1))
plt.title(f"Real: {int(age)} años | Estimación: {p_age:.1f} años")
plt.axis('off'); plt.show()